# RQ3: Yambda E5 Encoder Experiment

This notebook runs a quick encoder-comparison experiment for the E5 embeddings file.

Expected input from teammate:

```text
embeddings_e5.parquet
```

The notebook reuses old Yambda interactions if available. If raw interactions are missing, it downloads `yandex/yambda` likes.

## Do We Need Old Yambda Data?

Old Yambda **interactions** are enough if you have one of these files:

```text
data/yambda/interactions.parquet
data/RQ_album_artist_anchor/yambda/subset_interactions.parquet
data/yambda_e5_raw/interactions.parquet
```

Old processed files like `dvae_train_items.parquet` are **not enough** for the new encoder, because they already contain old embeddings. For E5 we must rebuild the prepared dVAE/seqrec data with `embeddings_e5.parquet`.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path('/home/jupyter/project/semantic_ids')
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd().resolve()

os.chdir(PROJECT_DIR)
print('Project dir:', PROJECT_DIR)

RAW_DIR = PROJECT_DIR / 'data' / 'yambda_e5_raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDINGS_PATH = RAW_DIR / 'embeddings_e5.parquet'
INTERACTIONS_PATH = RAW_DIR / 'interactions.parquet'

print('Raw dir:', RAW_DIR)
print('Embeddings path:', EMBEDDINGS_PATH)
print('Interactions path:', INTERACTIONS_PATH)

In [ ]:
# Locate/copy embeddings_e5.parquet.
# Put the file into data/yambda_e5_raw/embeddings_e5.parquet, or let this cell copy it from common locations.

embedding_candidates = [
    EMBEDDINGS_PATH,
    PROJECT_DIR / 'embeddings_e5.parquet',
    PROJECT_DIR / 'data' / 'amazon' / 'embeddings_e5.parquet',
    PROJECT_DIR / 'data' / 'yambda' / 'embeddings_e5.parquet',
    Path('/home/jupyter/embeddings_e5.parquet'),
    Path('/home/jupyter/project/semantic_ids/embeddings_e5.parquet'),
    Path('/Users/ad-scherbakov/Downloads/embeddings_e5.parquet'),
]

src_embedding = next((p for p in embedding_candidates if p.exists()), None)
if src_embedding is None:
    raise FileNotFoundError(
        'Cannot find embeddings_e5.parquet. Upload/copy it to '
        f'{EMBEDDINGS_PATH}'
    )

if src_embedding.resolve() != EMBEDDINGS_PATH.resolve():
    shutil.copy2(src_embedding, EMBEDDINGS_PATH)

print('Using embeddings:', EMBEDDINGS_PATH)
print('Size MB:', round(EMBEDDINGS_PATH.stat().st_size / 1e6, 2))

In [ ]:
# Reuse old Yambda interactions if they exist. Otherwise download likes.parquet from yandex/yambda.

interaction_candidates = [
    INTERACTIONS_PATH,
    PROJECT_DIR / 'data' / 'yambda' / 'interactions.parquet',
    PROJECT_DIR / 'data' / 'RQ_album_artist_anchor' / 'yambda' / 'subset_interactions.parquet',
]

src_interactions = next((p for p in interaction_candidates if p.exists()), None)
if src_interactions is not None:
    if src_interactions.resolve() != INTERACTIONS_PATH.resolve():
        shutil.copy2(src_interactions, INTERACTIONS_PATH)
    print('Reused interactions:', src_interactions)
else:
    print('Old interactions not found. Downloading yandex/yambda likes.parquet...')
    from datasets import load_dataset

    ds = load_dataset(
        'yandex/yambda',
        data_dir='flat/5b',
        data_files='likes.parquet',
    )['train'].to_polars()
    ds.write_parquet(INTERACTIONS_PATH)

print('Using interactions:', INTERACTIONS_PATH)
print('Size MB:', round(INTERACTIONS_PATH.stat().st_size / 1e6, 2))

In [ ]:
# Quick schema check.

import polars as pl

emb = pl.read_parquet(EMBEDDINGS_PATH, n_rows=3)
inter = pl.read_parquet(INTERACTIONS_PATH, n_rows=3)

print('Embeddings columns:', emb.columns)
print(emb)
print('\nInteractions columns:', inter.columns)
print(inter)

## Smoke Test

This runs a tiny version: 1000 users, 5000 core items, 1 dVAE epoch, no seqrec. Run this first.

In [ ]:
smoke_cmd = [
    sys.executable,
    '-m', 'scripts.RQ_encoder_comparison.run_encoder_experiment',
    '--interactions', str(INTERACTIONS_PATH),
    '--embedding', f'e5={EMBEDDINGS_PATH}',
    '--test-interval-days', '7',
    '--work-data-dir', 'data/RQ_encoder_comparison/yambda',
    '--results-dir', 'results/RQ_encoder_comparison_yambda',
    '--stages', 'prepare,dvae,sid_metrics',
    '--max-users', '1000',
    '--max-core-items', '5000',
    '--num-epochs', '1',
]

print('+', ' '.join(smoke_cmd))
subprocess.run(smoke_cmd, check=True)

## Full Quick Run

This is the lightweight run: 5000 users, up to 30000 core items, 3 dVAE epochs, then seqrec.

In [ ]:
full_cmd = [
    sys.executable,
    '-m', 'scripts.RQ_encoder_comparison.run_encoder_experiment',
    '--interactions', str(INTERACTIONS_PATH),
    '--embedding', f'e5={EMBEDDINGS_PATH}',
    '--test-interval-days', '7',
    '--work-data-dir', 'data/RQ_encoder_comparison/yambda',
    '--results-dir', 'results/RQ_encoder_comparison_yambda',
    '--stages', 'prepare,dvae,sid_metrics,seqrec',
    '--max-users', '5000',
    '--max-core-items', '30000',
    '--num-epochs', '3',
]

print('+', ' '.join(full_cmd))
subprocess.run(full_cmd, check=True)

## Collect Results

In [ ]:
collect_cmd = [
    sys.executable,
    '-m', 'scripts.RQ_encoder_comparison.collect_results',
    '--results-dir', 'results/RQ_encoder_comparison_yambda',
]

print('+', ' '.join(collect_cmd))
subprocess.run(collect_cmd, check=True)

comparison_path = PROJECT_DIR / 'results' / 'RQ_encoder_comparison_yambda' / 'comparison.csv'
print('Saved:', comparison_path)

In [ ]:
# View result table.

import pandas as pd

pd.read_csv(PROJECT_DIR / 'results' / 'RQ_encoder_comparison_yambda' / 'comparison.csv')